# Sparse Matrix Factorization of the token embeddings (EMBER, Phase A)

Paper: *Don't Forget Your Embeddings: Robust Knowledge Erasure via Precise Editing of Embeddings*
(Suslik, Shafran, Geva — EMNLP Findings 2026, [arXiv:2606.03695](https://arxiv.org/abs/2606.03695)).

**The idea.** Existing unlearning methods (RMU, SNMF, CRISP, PISCES) edit MLP / residual-stream
weights and ignore the embedding layer, which is why the erased knowledge can be recovered by
adversarial prompting or a little fine-tuning. EMBER adds a *precise* edit of the token-embedding
matrix, found with Sparse Matrix Factorization.

**What this notebook does** — just the factorization step (`ember.train_mf_features`, embedding
track), unrolled so you can poke at it:

1. Tokenize the concept + neutral sentences → $V'$, the set of token ids that actually occur.
2. Build $A = E[V']^\top \in \mathbb{R}^{d_{model} \times |V'|}$ from the model's embedding matrix.
3. Factorize $A \approx F G^\top$ with alternating ridge least squares + winner-take-all sparsity:
   - $F \in \mathbb{R}^{d_{model} \times K}$ — **dense, signed** directions in model space (what EMBER subtracts).
   - $G \in \mathbb{R}^{|V'| \times K}$ — **WTA-sparse, signed** per-token scores (each feature keeps its top `g_sparsity` fraction of tokens).
4. Score every feature with `ratio_abs = mean|G| over concept tokens / mean|G| over neutral tokens`
   — this is the number Phase B (`interpret_features.py`) thresholds before asking Gemini
   "is this feature the concept?".

Nothing here is erased or evaluated; that is Phase C (`demo.ipynb` / `ember.run_erasure`).

## 1. Configuration

These are the **standard parameters** used by the paper's pipeline for Gemma
(`--ranks 100 --seed 42`, `g_sparsity=0.01`, `max_iterations=20000`).
For Llama-3.1-8B-Instruct the paper uses `RANK = 200`.

`CONCEPT` must be one of the 18 concepts in `data/concept_sentences.json` (printed below).

In [1]:
import json
import os
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv

load_dotenv()  # HF_TOKEN (needed for gated models such as Llama)

# ---- experiment knobs -------------------------------------------------
CONCEPT = "Harry Potter"
MODEL_NAME = "google/gemma-2-2b-it"   # or "meta-llama/Llama-3.1-8B-Instruct"
RANK = 100                             # 100 for Gemma, 200 for Llama
SEED = 42
G_SPARSITY = 0.01                      # WTA: fraction of |V'| tokens kept per feature
MAX_ITERATIONS = 20_000
K_PROJ = 30                            # how many logit-lens tokens to keep per feature
OUTDIR = "mf_outputs"                  # only used if you run the save cell at the end

pd.set_option("display.max_colwidth", 160)

concepts = [e["concept"] for e in json.loads(Path("data/concept_sentences.json").read_text())]
print(f"available concepts ({len(concepts)}):")
print("  " + "\n  ".join(concepts))
assert CONCEPT in concepts, f"{CONCEPT!r} is not in data/concept_sentences.json"

available concepts (18):
  Culture of Greece
  Golf
  Republic of Ireland
  Ancient Rome
  Baseball
  Uranium
  Harry Potter
  Cannabis
  Gambling
  Gun
  Pornography
  Valentine's Day
  World War II
  Nazism
  Heroin
  Halloween
  Artificial intelligence
  COVID-19 pandemic


In [2]:
from llm_utils.activation_generator import (
    ActivationGenerator,
    extract_token_ids_sample_ids_and_labels,
)
from transformers import AutoTokenizer

from ember.local_datasets import ConceptDataset
from ember.utils import (
    SparseMatrixFactorization,
    build_token_label_codes,
    collect_feature_rows_for_embeddings,
    compute_embedding_stats,
    fit_with_ridge,
    get_embedding_matrix,
    get_pipeline_path,
    get_special_token_ids,
    resolve_device,
    save_df_to_csv,
    set_seed,
    vector_to_logits,
    _safe_concept,
    _safe_model_name,
    _safe_tokens,
)

set_seed(SEED)
MODEL_DEVICE = resolve_device("auto")
FIT_DEVICE = resolve_device("auto")
print(f"model device: {MODEL_DEVICE} | fitting device: {FIT_DEVICE}")

/homes/orip/.conda/envs/Ember/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


model device: cuda | fitting device: cuda


## 2. Load the model

Loaded through TransformerLens (`HookedTransformer`). The embedding track only needs `W_E`
(to build $A$), `W_U` (to read features back out as tokens) and the tokenizer — no forward passes.

In [3]:
act_generator = ActivationGenerator(MODEL_NAME, model_device=MODEL_DEVICE, data_device="cpu", mode="mlp")
model = act_generator.model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

E = get_embedding_matrix(model)  # (|V|, d_model)
print(f"W_E: {tuple(E.shape)}  dtype={E.dtype}")

Loading checkpoint shards: 100%|██████████| 2/2 [00:05<00:00,  2.68s/it]


RuntimeError: CUDA error: operation not supported
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## 3. Build the matrix $A$

$V'$ = unique, non-special token ids appearing in the concept sentences **or** the neutral sentences.
`neutral_sample_seed=42` is fixed in the pipeline so that $V'$ is reproducible.

In [ ]:
dataset = ConceptDataset(concept_name=CONCEPT, neutral_sample_seed=42)
token_ids_all, sample_ids, labels_all = extract_token_ids_sample_ids_and_labels(dataset, act_generator)

special = get_special_token_ids(model)
vprime_token_ids = sorted(int(t) for t in set(token_ids_all.tolist()) if int(t) not in special)

E_vprime = E.index_select(0, torch.tensor(vprime_token_ids, dtype=torch.long, device=E.device)).detach()
A_embed = E_vprime.T.contiguous()  # (d_model, |V'|)

n_concept_sents = sum(1 for _, lbl in dataset if lbl == CONCEPT)
print(f"sentences: {len(dataset)} ({n_concept_sents} concept / {len(dataset) - n_concept_sents} neutral)")
print(f"tokens seen: {len(token_ids_all)} -> |V'| = {len(vprime_token_ids)}")
print(f"A = E[V'].T : {tuple(A_embed.shape)}  (d_model x |V'|)")
print(f"tokens kept per feature by WTA: {max(1, int(G_SPARSITY * len(vprime_token_ids)))}")

## 4. Fit the factorization

`SparseMatrixFactorization.fit` alternates two closed-form ridge least-squares updates

$$F \leftarrow A G (G^\top G + \lambda I)^{-1}, \qquad G \leftarrow A^\top F (F^\top F + \lambda I)^{-1}$$

and applies `wta_cols(G, pct_keep=g_sparsity)` after each $G$ update (keeps the top `g_sparsity`
fraction of $|G_{:,k}|$, zeroes the rest). $F$ stays dense. Early stopping on the Frobenius loss
with `patience=500`. `fit_with_ridge` retries with `reg` ×10 if the inverse is singular.

This is the slow cell (minutes on a GPU).

In [ ]:
from ember.timing import Timer

set_seed(SEED)
A_fit = A_embed.to(FIT_DEVICE).float()

nmf = SparseMatrixFactorization(RANK, fitting_device=FIT_DEVICE, g_sparsity=G_SPARSITY)
with Timer() as t:
    reg_used = fit_with_ridge(nmf, A_fit, MAX_ITERATIONS)
print(f"\nfitted in {t['elapsed']:.1f}s with reg={reg_used:g}")

F_emb = torch.as_tensor(nmf.F_).detach().cpu()  # (d_model, K)
G_tok = torch.as_tensor(nmf.G_).detach().cpu()  # (|V'|, K)
print(f"F: {tuple(F_emb.shape)}   G: {tuple(G_tok.shape)}")

In [ ]:
# --- fit diagnostics ---------------------------------------------------
with torch.no_grad():
    A_hat = (F_emb @ G_tok.T).to(A_fit.device)
    resid = torch.norm(A_fit - A_hat, p="fro").item()
    rel = resid / torch.norm(A_fit, p="fro").item()

nz_per_feature = (G_tok.abs() > 1e-12).sum(0)
nz_per_token = (G_tok.abs() > 1e-12).sum(1)

print(f"||A - FG^T||_F        = {resid:.3f}")
print(f"relative error        = {rel:.4f}  ({100 * (1 - rel):.1f}% of the Frobenius norm explained)")
print(f"G density             = {(G_tok.abs() > 1e-12).float().mean().item():.4f}")
print(f"nonzero tokens/feature: min={nz_per_feature.min().item()} "
      f"median={nz_per_feature.median().item()} max={nz_per_feature.max().item()}")
print(f"features/token        : mean={nz_per_token.float().mean().item():.2f} "
      f"max={nz_per_token.max().item()}, {int((nz_per_token == 0).sum())} tokens covered by no feature")
print(f"||F_:,k||_2           : min={F_emb.norm(dim=0).min().item():.3f} "
      f"max={F_emb.norm(dim=0).max().item():.3f}")

## 5. Per-feature concept-vs-neutral statistics

Each token id is labelled by *where it was seen*: `1` concept-only, `0` neutral-only, `2` both.
`compute_embedding_stats` then compares $\overline{|G_{:,k}|}$ over concept-only tokens against
neutral-only tokens (`both` tokens are counted but excluded from the means).

**`ratio_abs` is the selection score.** Phase B keeps features with `ratio_abs >= 2.0` by default.

In [ ]:
token_to_code = build_token_label_codes(CONCEPT, dataset, tokenizer)
df_stats = compute_embedding_stats(G_tok, vprime_token_ids, token_to_code, CONCEPT)

df_top = df_stats.sort_values("ratio_abs", ascending=False)
df_top.head(15)[
    ["feature", "ratio_abs", "diff_abs", "mean_abs_concept", "mean_abs_neutral",
     "num_concept", "num_neutral", "num_both"]
]

In [ ]:
# How many features survive each selection threshold?
for thr in [1.0, 1.25, 1.5, 1.75, 2.0, 3.0, 5.0]:
    n = int((df_stats["ratio_abs"] >= thr).sum())
    print(f"ratio_abs >= {thr:<5} : {n:3d}/{len(df_stats)} features ({100 * n / len(df_stats):.0f}%)")

## 6. What is each feature *about*?

Two complementary readouts, exactly as `train_mf_features.py` writes them into `token_features.csv`:

- **activating tokens** — nonzero entries of $G_{:,k}$, sorted by $|G|$. "Which tokens use this direction."
- **projection tokens** — logit lens: $F_{:,k}$ pushed through `ln_final` + `W_U`. "What this direction writes."

In [ ]:
token_label_map = {
    tid: "both" if c == 2 else CONCEPT if c == 1 else "Neutral"
    for tid, c in token_to_code.items()
}
rows = collect_feature_rows_for_embeddings(
    G_tok, vprime_token_ids, token_label_map, model, _safe_model_name(MODEL_NAME), CONCEPT, RANK
)

# logit lens: F columns are already d_model directions
V_embed = F_emb.T.to(device=E_vprime.device, dtype=E_vprime.dtype)  # (K, d_model)
logits_embed = vector_to_logits(model, V_embed)
topk = torch.topk(logits_embed, K_PROJ, dim=-1)
bottomk = torch.topk(logits_embed, K_PROJ, dim=-1, largest=False)
abs_topk = torch.topk(logits_embed.abs(), K_PROJ * 2, dim=-1)

def _tok(idx):
    return _safe_tokens(model.to_str_tokens(torch.tensor(idx.tolist(), dtype=torch.long)))


for k, row in enumerate(rows):
    row["projection_top_tokens"] = _tok(topk.indices[k])
    row["projection_bottom_tokens"] = _tok(bottomk.indices[k])
    row["projection_abs_top_tokens"] = _tok(abs_topk.indices[k])

df_feats = pd.DataFrame(rows).merge(df_stats[["feature", "ratio_abs", "diff_abs"]], on="feature")
print(f"{int(df_feats['is_concept_related'].sum())}/{len(df_feats)} features are >=70% concept tokens")
df_feats.head()

In [ ]:
def show_feature(k: int, n_tokens: int = 25) -> None:
    row = df_feats.loc[df_feats["feature"] == k].iloc[0]
    print(f"=== feature {k}  ratio_abs={row['ratio_abs']:.2f}  "
          f"concept_tokens={row['num_concept_related']}/{row['num_activating_tokens_all']} ===")
    print("  activating :", " ".join(repr(t) for t in row["activating_tokens"][:n_tokens]))
    print("  proj top   :", " ".join(repr(t) for t in row["projection_top_tokens"][:n_tokens]))
    print("  proj bottom:", " ".join(repr(t) for t in row["projection_bottom_tokens"][:n_tokens]))
    print()


for k in df_top["feature"].head(8):
    show_feature(int(k))

In [ ]:
# Contrast: the lowest-ratio features should look generic / neutral.
for k in df_top["feature"].tail(3):
    show_feature(int(k))

## 7. Which tokens would EMBER touch?

The paper's coherence argument: the cost of the edit is *localized* to a small set of
concept-exclusive tokens. Take the features that pass the threshold and look at the union of
their supports.

In [ ]:
RATIO_THRESH = 2.0
selected = df_stats.loc[df_stats["ratio_abs"] >= RATIO_THRESH, "feature"].astype(int).tolist()

support = (G_tok[:, selected].abs() > 1e-12).any(dim=1)
hit_ids = [vprime_token_ids[i] for i in torch.where(support)[0].tolist()]
codes = np.array([token_to_code.get(t, 0) for t in hit_ids])

print(f"selected features: {len(selected)}/{RANK}")
print(f"tokens in their union support: {len(hit_ids)} / {len(vprime_token_ids)} "
      f"({100 * len(hit_ids) / len(vprime_token_ids):.1f}% of V')")
print(f"  concept-only {int((codes == 1).sum())} | both {int((codes == 2).sum())} | neutral-only {int((codes == 0).sum())}")
print("\nconcept-only tokens in the support:")
print(_safe_tokens(model.to_str_tokens(torch.tensor([t for t, c in zip(hit_ids, codes) if c == 1][:60]))))

## 8. (Optional) Save in the pipeline's on-disk layout

Writes the same three artifacts `ember.train_mf_features` produces, so that
`ember.interpret_features` and `ember.run_erasure --features-source local` can pick them up:

```
mf_outputs/<model>/pickles/rank<R>/seed<S>/<concept>/embedding/embedding.pkl
mf_outputs/<model>/csvs/rank<R>/seed<S>/<concept>/embedding/token_features.csv
mf_outputs/<model>/csvs/rank<R>/seed<S>/<concept>/embedding/stats_embed.csv
```

Note `save_df_to_csv` **appends** to an existing CSV (deduping on `feature`), so delete the
directory first if you are overwriting a previous experiment.

In [ ]:
SAVE = False  # flip to True to write to mf_outputs/

if SAVE:
    safe_model, safe_concept = _safe_model_name(MODEL_NAME), _safe_concept(CONCEPT)
    pkl_path = get_pipeline_path(OUTDIR, safe_model, "pickles", RANK, safe_concept,
                                 "embedding", "embedding.pkl", seed=SEED)
    csv_path = get_pipeline_path(OUTDIR, safe_model, "csvs", RANK, safe_concept,
                                 "embedding", "token_features.csv", seed=SEED)
    stats_path = get_pipeline_path(OUTDIR, safe_model, "csvs", RANK, safe_concept,
                                   "embedding", "stats_embed.csv", seed=SEED)

    with open(pkl_path, "wb") as f:
        pickle.dump({"nmf": nmf, "vprime_token_ids": vprime_token_ids}, f)
    save_df_to_csv(pd.DataFrame(rows), csv_path, dedupe_cols=["feature"])
    save_df_to_csv(df_stats, stats_path)
    print("wrote:", pkl_path, csv_path, stats_path, sep="\n  ")
else:
    print("SAVE=False — nothing written")

## 9. (Optional) Sweep rank / sparsity

Everything above reuses `A_embed`, so a sweep only re-runs the fit. Expect a trade-off:
higher rank and higher `g_sparsity` lower the reconstruction error, but make individual
features less concept-exclusive (fewer features pass `ratio_abs >= 2`).

In [ ]:
def fit_summary(rank: int, g_sparsity: float, max_iter: int = 5_000) -> dict:
    set_seed(SEED)
    m = SparseMatrixFactorization(rank, fitting_device=FIT_DEVICE, g_sparsity=g_sparsity)
    with Timer() as timer:
        fit_with_ridge(m, A_fit, max_iter)
    F = torch.as_tensor(m.F_).detach().cpu()
    G = torch.as_tensor(m.G_).detach().cpu()
    rel_err = (torch.norm(A_fit - (F @ G.T).to(A_fit.device), p="fro")
               / torch.norm(A_fit, p="fro")).item()
    stats = compute_embedding_stats(G, vprime_token_ids, token_to_code, CONCEPT)
    return {
        "rank": rank,
        "g_sparsity": g_sparsity,
        "rel_err": rel_err,
        "n_ratio_ge_2": int((stats["ratio_abs"] >= 2.0).sum()),
        "max_ratio_abs": float(stats["ratio_abs"].max()),
        "seconds": timer["elapsed"],
    }


sweep = pd.DataFrame([
    fit_summary(rank, g)
    for rank in (50, 100, 200)
    for g in (0.005, 0.01, 0.02)
])
sweep